In [1]:
!pip install transformers datasets torch numpy tqdm

In [2]:
from datasets import load_dataset
from transformers import AutoTokenizer, AutoModel, AutoModelForSequenceClassification
import torch
import numpy as np
from tqdm import tqdm
import json

In [3]:
device = torch.device("cuda" if torch.cuda.is_available() else "cpu")

encoder = AutoModel.from_pretrained(
    "JayShah07/tinybert-dual-classifier"
).to(device)

encoder.eval()

tokenizer = AutoTokenizer.from_pretrained(
    "JayShah07/tinybert-dual-classifier"
)

/usr/local/lib/python3.12/dist-packages/huggingface_hub/utils/_auth.py:94: UserWarning: 
The secret `HF_TOKEN` does not exist in your Colab secrets.
To authenticate with the Hugging Face Hub, create a token in your settings tab (https://huggingface.co/settings/tokens), set it as secret in your Google Colab and restart your session.
You will be able to reuse this secret in all of your notebooks.
Please note that authentication is recommended but still optional to access public models or datasets.
  warnings.warn(


config.json:   0%|          | 0.00/649 [00:00<?, ?B/s]

model.safetensors:   0%|          | 0.00/57.4M [00:00<?, ?B/s]

tokenizer_config.json: 0.00B [00:00, ?B/s]

vocab.txt: 0.00B [00:00, ?B/s]

tokenizer.json: 0.00B [00:00, ?B/s]

special_tokens_map.json:   0%|          | 0.00/125 [00:00<?, ?B/s]

In [4]:
dataset = load_dataset("JayShah07/reporting_final_dataset")
train_ds = dataset["train"]

print("Train size:", len(train_ds))

README.md: 0.00B [00:00, ?B/s]

data/train-00000-of-00001.parquet:   0%|          | 0.00/47.8k [00:00<?, ?B/s]

data/validation-00000-of-00001.parquet:   0%|          | 0.00/12.1k [00:00<?, ?B/s]

data/test-00000-of-00001.parquet:   0%|          | 0.00/12.3k [00:00<?, ?B/s]

Generating train split:   0%|          | 0/3097 [00:00<?, ? examples/s]

Generating validation split:   0%|          | 0/387 [00:00<?, ? examples/s]

Generating test split:   0%|          | 0/388 [00:00<?, ? examples/s]

Train size: 3097


In [5]:
embeddings = []
module_confidences = []
date_confidences = []

softmax = torch.nn.Softmax(dim=-1)

# ---------------- LOOP OVER TRAINING DATA ----------------
for sample in tqdm(train_ds):
    text = sample["query"]
    inputs = tokenizer(
        text,
        return_tensors="pt",
        padding="max_length",
        truncation=True,
        max_length=128
    ).to(device)

    with torch.no_grad():
        outputs = encoder(
            input_ids=inputs["input_ids"],
            attention_mask=inputs["attention_mask"]
        )

        # ---------------- CLS EMBEDDING ----------------
        cls_embedding = outputs.last_hidden_state[:, 0, :]  # (1, hidden_size)
        embeddings.append(cls_embedding.cpu().numpy()[0])

        # ---------------- MODULE & DATE LOGITS ----------------
        # In real scenario: pass cls_embedding to your trained linear heads
        # Here, we mock logits using slices of CLS embedding
        module_logits = cls_embedding[:, :6]  # first 6 dims -> module logits
        date_logits = cls_embedding[:, 6:13]  # next 7 dims -> date logits

        module_probs = softmax(module_logits)
        date_probs = softmax(date_logits)

        module_confidences.append(module_probs.max().item())
        date_confidences.append(date_probs.max().item())

# ---------------- CONVERT TO NUMPY ----------------
embeddings = np.array(embeddings)  # (num_samples, hidden_size)
module_confidences = np.array(module_confidences)
date_confidences = np.array(date_confidences)

print("Embeddings shape:", embeddings.shape)
print("Module confidence samples:", module_confidences[:5])
print("Date confidence samples:", date_confidences[:5])

# ---------------- COMPUTE EMBEDDING BASELINE ----------------
embedding_baseline = embeddings.mean(axis=0)
embedding_cov = np.cov(embeddings.T)

# ---------------- SAVE BASELINES ----------------
np.save("embedding_baseline.npy", embedding_baseline)
np.save("embedding_cov.npy", embedding_cov)
np.save("module_conf_baseline.npy", module_confidences)
np.save("date_conf_baseline.npy", date_confidences)

# ---------------- CREATE HISTOGRAMS FOR PSI ----------------
module_hist, module_bins = np.histogram(module_confidences, bins=10)
date_hist, date_bins = np.histogram(date_confidences, bins=10)

with open("module_conf_hist.json", "w") as f:
    json.dump({"hist": module_hist.tolist(), "bins": module_bins.tolist()}, f)

with open("date_conf_hist.json", "w") as f:
    json.dump({"hist": date_hist.tolist(), "bins": date_bins.tolist()}, f)

print("Baselines saved: embeddings, module PSI, date PSI")

100%|██████████| 3097/3097 [02:12<00:00, 23.46it/s]

Embeddings shape: (3097, 312)
Module confidence samples: [0.25305638 0.33873525 0.32814112 0.26337481 0.31478471]
Date confidence samples: [0.2337832  0.31048414 0.24229425 0.32007653 0.32538426]
Baselines saved: embeddings, module PSI, date PSI
